In [ ]:
"""
LAI Estimation Pipeline — K-Nearest Neighbors (KNN), with Recursive Feature
Elimination (RFE)
Source domain: Rema-Kalenga Wildlife Sanctuary
Spatial transfer target: Khadimnagar National Park (KNP)

RFE is applied AFTER hyperparameter tuning, using the tuned KNN as the
evaluator. KNN has no native feature_importances_ or coefficients (purely
instance-based), so elimination uses permutation_importance computed on the
training set only.

RFE runs via LOOCV on the 32-plot Rema-Kalenga TRAINING SET ONLY — the 8-plot
held-out test set and the KNP spatial-transfer target are never touched during
feature selection, avoiding data leakage.

From the RFE step onward, rfe_selected_features_knn (not the full 15-feature
feature_cols) is used for the final model fit, test evaluation, and transfer.

IMPORTANT: KNN is distance-based and therefore sensitive to feature scale. A
StandardScaler is fit on the training data only (on the RFE-selected columns,
at final-fit time) and reused for the test set and the KNP transfer target.

NOTE ON SAMPLE SIZE: With only 32 training plots, LOOCV means each fold trains
on 31 points. KNN's 'n_neighbors' parameter must therefore stay well below 31 —
the grid below caps at 10, which is already a large fraction of the training set.
"""

# ============================================================
# SETUP
# ============================================================
get_ipython().system('pip install earthengine-api geemap -q')

import ee
ee.Authenticate()
ee.Initialize(project='osmgee')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import itertools
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


# ============================================================
# REMA-KALENGA: BOUNDARY AND SITE LOCATIONS (SOURCE / TRAINING DATA)
# ============================================================
rema_boundary = ee.FeatureCollection('projects/osmgee/assets/RemaKalenga')
rema_sites = ee.FeatureCollection('projects/osmgee/assets/RemaSites')

print('Rema-Kalenga boundary features:', rema_boundary.size().getInfo())
print('Rema-Kalenga sites features:', rema_sites.size().getInfo())

rema_first_feature = rema_sites.first()
print('Available fields:', rema_first_feature.propertyNames().getInfo())

rema_sites_info = rema_sites.select(
    ['ClusterID', 'MEAN_PAR_L', 'XCoord', 'YCoord', 'longitude', 'Llllllatit']
).getInfo()

for f in rema_sites_info['features']:
    print(f['properties'])


# ============================================================
# CLEAN FIELD NAMES (XCoord/Llllllatit = longitude, YCoord/longitude = latitude)
# ============================================================
def clean_feature_rema(f):
    lon = ee.Number(f.get('XCoord'))
    lat = ee.Number(f.get('YCoord'))
    par_lai = ee.Number(f.get('MEAN_PAR_L'))
    cluster_id = f.get('ClusterID')

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'cluster_id': cluster_id,
            'par_lai': par_lai,
            'longitude': lon,
            'latitude': lat
        }
    )

rema_sites_clean = rema_sites.map(clean_feature_rema)

print('Cleaned feature count:', rema_sites_clean.size().getInfo())
print('Sample feature:', rema_sites_clean.first().getInfo())


# ============================================================
# SENTINEL-2: LOAD AND FILTER FOR REMA-KALENGA FIELD WINDOW
# ============================================================
rema_field_start = ee.Date('2021-01-01')
rema_field_end = ee.Date('2021-03-31')

rema_s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(rema_boundary) \
    .filterDate(rema_field_start, rema_field_end) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

print('Number of available Sentinel-2 scenes (Rema-Kalenga):', rema_s2_collection.size().getInfo())

rema_image_list = rema_s2_collection.toList(rema_s2_collection.size())
n_rema = rema_s2_collection.size().getInfo()
for i in range(n_rema):
    img = ee.Image(rema_image_list.get(i))
    date = img.date().format('YYYY-MM-dd').getInfo()
    cloud = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    print(f'{date}  —  cloud cover: {cloud:.1f}%')


# ============================================================
# CLOUD MASK, COMPOSITE, AND SPECTRAL INDICES
# ============================================================
csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
QA_BAND = 'cs_cdf'
CLEAR_THRESHOLD = 0.60

def mask_s2_clouds(image):
    return image.updateMask(image.select(QA_BAND).gte(CLEAR_THRESHOLD)) \
                .divide(10000).copyProperties(image, image.propertyNames())

rema_s2_masked = rema_s2_collection.linkCollection(csPlus, [QA_BAND]).map(mask_s2_clouds)
rema_s2_composite = rema_s2_masked.median().clip(rema_boundary)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('EVI')

    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')

    msavi = image.expression(
        '(2 * NIR + 1 - sqrt((2 * NIR + 1)**2 - 8 * (NIR - RED))) / 2', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('MSAVI')

    return image.addBands([ndvi, gndvi, evi, savi, msavi])

rema_s2_with_indices = add_indices(rema_s2_composite)

raw_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
index_bands = ['NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']

final_image_rema = rema_s2_with_indices.select(raw_bands + index_bands)
print('Bands in Rema-Kalenga composite:', final_image_rema.bandNames().getInfo())


# ============================================================
# BUILD 15m x 15m SQUARE PLOTS AND EXTRACT MEAN SPECTRAL VALUES
# ============================================================
def make_square_plot(feature):
    point = feature.geometry()
    half_side = 7.5
    square = point.buffer(half_side, 1).bounds()
    return feature.setGeometry(square)

rema_sites_squares = rema_sites_clean.map(make_square_plot)
sample_geom = rema_sites_squares.first().geometry().getInfo()
print('Sample square plot geometry (Rema-Kalenga):', sample_geom)

rema_extracted = final_image_rema.reduceRegions(
    collection=rema_sites_squares,
    reducer=ee.Reducer.mean(),
    scale=10
)

rema_extracted_info = rema_extracted.getInfo()

for f in rema_extracted_info['features'][:5]:
    print(f['properties'])
    print('---')

rema_rows = [f['properties'] for f in rema_extracted_info['features']]
df_rema = pd.DataFrame(rema_rows)

id_cols = ['cluster_id', 'par_lai', 'longitude', 'latitude']
other_cols = [c for c in df_rema.columns if c not in id_cols]
df_rema = df_rema[id_cols + other_cols]
df_rema


# ============================================================
# TRAIN / TEST SPLIT (80/20 -> 32/8)
# ============================================================
feature_cols = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
                 'NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']
target_col = 'par_lai'

X_rema = df_rema[feature_cols]
y_rema = df_rema[target_col]

X_train_rema, X_test_rema, y_train_rema, y_test_rema = train_test_split(
    X_rema, y_rema, test_size=0.2, random_state=42
)

print('Train PAR LAI range:', round(y_train_rema.min(), 2), '-', round(y_train_rema.max(), 2))
print('Test PAR LAI range:', round(y_test_rema.min(), 2), '-', round(y_test_rema.max(), 2))
print('Train PAR LAI mean:', round(y_train_rema.mean(), 2))
print('Test PAR LAI mean:', round(y_test_rema.mean(), 2))


# ============================================================
# FEATURE SCALING (required for KNN — fit on FULL 15-feature train set here,
# used only for the hyperparameter search and RFE stages below; the scaler is
# refit on the RFE-selected columns at final-fit time, see that section)
# ============================================================
scaler_full = StandardScaler()
X_train_rema_scaled = pd.DataFrame(
    scaler_full.fit_transform(X_train_rema), columns=feature_cols, index=X_train_rema.index
)

print('Feature scaling applied (StandardScaler fit on 32 training plots only).')
print('Scaled train feature means (should be ~0):')
print(X_train_rema_scaled.mean().round(3))


# ============================================================
# KNN — HYPERPARAMETER SEARCH VIA LOOCV
# ============================================================
knn_param_grid = {
    'n_neighbors': [2, 3, 4, 5, 6, 7, 8, 10],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_keys = list(knn_param_grid.keys())
knn_combinations = list(itertools.product(*knn_param_grid.values()))
print(f'Total KNN hyperparameter combinations to test: {len(knn_combinations)}')

loo = LeaveOneOut()
knn_results = []

for combo in knn_combinations:
    params = dict(zip(knn_keys, combo))
    model = KNeighborsRegressor(**params)

    scores = cross_val_score(model, X_train_rema_scaled, y_train_rema, cv=loo, scoring='neg_mean_squared_error')
    mean_mse = -scores.mean()
    rmse = np.sqrt(mean_mse)

    knn_results.append({**params, 'loocv_rmse': rmse})

knn_results_df = pd.DataFrame(knn_results).sort_values('loocv_rmse').reset_index(drop=True)
print('Top 5 KNN hyperparameter combinations by LOOCV RMSE:')
print(knn_results_df.head(5))


# ============================================================
# EXTRACT BEST HYPERPARAMETERS
# ============================================================
best_params_knn = knn_results_df.iloc[0][knn_keys].to_dict()
best_params_knn['n_neighbors'] = int(best_params_knn['n_neighbors'])

print('Best hyperparameters selected via LOOCV (KNN, Rema-Kalenga):')
print(best_params_knn)


def compute_metrics(y_true, y_pred, label):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    rrmse = (rmse / y_true.mean()) * 100
    bias = (y_pred - y_true).mean()
    return {
        'Set': label,
        'N': len(y_true),
        'R2': round(r2, 3),
        'RMSE': round(rmse, 3),
        'MAE': round(mae, 3),
        'rRMSE (%)': round(rrmse, 1),
        'Bias (mean residual)': round(bias, 3)
    }


# ============================================================
# RECURSIVE FEATURE ELIMINATION (RFE) — KNN
# Runs AFTER hyperparameter tuning, using the tuned KNN as the evaluator.
# KNN has no native feature_importances_/coef_ (purely instance-based), so
# elimination is driven by permutation_importance (scored by RMSE increase
# when a feature is shuffled).
# Operates ONLY on the 32-plot training set via LOOCV/permutation on training
# data — the 8-plot test set and the KNP spatial-transfer target are never
# touched here.
# ============================================================
min_features = 3                 # safety floor given only 32 training samples
elimination_step = 1              # remove 1 feature per iteration
absolute_drop_threshold = 0.15    # stop if LOOCV RMSE rises by more than this (LAI units)

current_features_knn = feature_cols.copy()
rfe_results_knn = []
iteration = 0

print('\n' + '='*60)
print('STARTING RFE — KNN (Rema-Kalenga training set, LOOCV)')
print('='*60)

start_time = time.time()

while True:
    iteration += 1
    n_current = len(current_features_knn)

    # Re-scale using only the current feature subset (scaling depends on which
    # columns are present)
    scaler_iter = StandardScaler()
    X_train_iter_scaled = scaler_iter.fit_transform(X_train_rema[current_features_knn])

    knn_eval = KNeighborsRegressor(**best_params_knn)
    scores = cross_val_score(
        knn_eval,
        X_train_iter_scaled,
        y_train_rema,
        cv=loo,
        scoring='neg_mean_squared_error'
    )
    rmse = np.sqrt(-scores.mean())

    if len(rfe_results_knn) == 0:
        baseline_rmse = rmse
        rmse_increase = 0.0
    else:
        baseline_rmse = rfe_results_knn[0]['rmse']
        rmse_increase = rmse - baseline_rmse

    print(f'\nIteration {iteration}: {n_current} features -> LOOCV RMSE = {rmse:.3f} '
          f'(change from baseline: {rmse_increase:+.3f})')

    rfe_results_knn.append({
        'iteration': iteration,
        'n_features': n_current,
        'features': current_features_knn.copy(),
        'rmse': rmse,
        'rmse_increase': rmse_increase
    })

    should_stop = False
    if len(rfe_results_knn) > 1 and rmse_increase > absolute_drop_threshold:
        should_stop = True
        print(f'  STOPPING: RMSE rose by {rmse_increase:.3f} (threshold: {absolute_drop_threshold})')
    if n_current <= min_features:
        should_stop = True
        print(f'  STOPPING: reached minimum feature count ({min_features})')

    if should_stop:
        break

    # Fit on full training subset, then compute permutation importance to rank features
    # Note: n_neighbors must not exceed the number of samples used for fitting.
    safe_k = min(best_params_knn['n_neighbors'], len(X_train_iter_scaled) - 1)
    knn_importance = KNeighborsRegressor(
        n_neighbors=safe_k, weights=best_params_knn['weights'], metric=best_params_knn['metric']
    )
    knn_importance.fit(X_train_iter_scaled, y_train_rema)

    perm_result = permutation_importance(
        knn_importance, X_train_iter_scaled, y_train_rema,
        n_repeats=30, random_state=42, scoring='neg_mean_squared_error'
    )
    importances = dict(zip(current_features_knn, perm_result.importances_mean))

    sorted_feats = sorted(importances.items(), key=lambda x: x[1])
    n_to_remove = min(elimination_step, len(current_features_knn) - min_features)
    features_to_remove = [f for f, _ in sorted_feats[:n_to_remove]]
    print(f'  Removing: {features_to_remove}')

    current_features_knn = [f for f in current_features_knn if f not in features_to_remove]

total_time = time.time() - start_time

best_rfe_knn = min(rfe_results_knn, key=lambda x: x['rmse'])
rfe_selected_features_knn = best_rfe_knn['features']

print('\n' + '='*60)
print('RFE COMPLETE — KNN')
print('='*60)
print(f'Total time: {total_time/60:.1f} min')
print(f'Optimal feature count: {best_rfe_knn["n_features"]}')
print(f'Optimal features: {rfe_selected_features_knn}')
print(f'Optimal LOOCV RMSE: {best_rfe_knn["rmse"]:.3f} (baseline was {rfe_results_knn[0]["rmse"]:.3f})')

# --- Plot: LOOCV RMSE vs number of features ---
rfe_df_knn = pd.DataFrame(rfe_results_knn)

plt.figure(figsize=(9, 6))
plt.plot(rfe_df_knn['n_features'], rfe_df_knn['rmse'], marker='o', markersize=8,
         linewidth=2.5, color='#2E86AB', label='LOOCV RMSE')

best_idx_knn = rfe_df_knn['rmse'].idxmin()
plt.scatter(rfe_df_knn.loc[best_idx_knn, 'n_features'], rfe_df_knn.loc[best_idx_knn, 'rmse'],
            color='gold', s=300, marker='*', zorder=5, edgecolor='black',
            label=f'Optimal: {rfe_df_knn.loc[best_idx_knn, "n_features"]} features')

plt.xlabel('Number of Features')
plt.ylabel('LOOCV RMSE (PAR LAI)')
plt.title('Recursive Feature Elimination (KNN): RMSE vs Feature Count\n(Rema-Kalenga training set, LOOCV)')
plt.gca().invert_xaxis()
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# REFIT FINAL MODEL ON RFE-SELECTED FEATURES — THIS IS THE MODEL USED FOR TRANSFER
# A fresh scaler is fit on the RFE-selected columns only (train set), and
# reused (never refit) for the test set and the KNP transfer target.
# ============================================================
scaler = StandardScaler()
X_train_rema_final_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_rema[rfe_selected_features_knn]),
    columns=rfe_selected_features_knn, index=X_train_rema.index
)
X_test_rema_final_scaled = pd.DataFrame(
    scaler.transform(X_test_rema[rfe_selected_features_knn]),
    columns=rfe_selected_features_knn, index=X_test_rema.index
)

final_knn_rema = KNeighborsRegressor(**best_params_knn)
final_knn_rema.fit(X_train_rema_final_scaled, y_train_rema)

y_pred_train_rema_knn = final_knn_rema.predict(X_train_rema_final_scaled)
y_pred_test_rema_knn = final_knn_rema.predict(X_test_rema_final_scaled)

metrics_train_rema_knn = compute_metrics(y_train_rema, y_pred_train_rema_knn, 'Rema-Kalenga Train (in-sample, KNN, RFE features)')
metrics_test_rema_knn = compute_metrics(y_test_rema, y_pred_test_rema_knn, 'Rema-Kalenga Test (held-out, KNN, RFE features)')

metrics_df_rema_knn = pd.DataFrame([metrics_train_rema_knn, metrics_test_rema_knn])
print('=== Rema-Kalenga KNN model performance summary (RFE-selected features) ===')
print(metrics_df_rema_knn.to_string(index=False))
print()

print('=== Feature importance: not available for KNN ===')
print('KNN has no native feature importance (instance-based, not weight-based).')
print('(Permutation importance was used internally to drive RFE elimination.)')
print()

residuals_df_rema_knn = pd.DataFrame({
    'cluster_id': df_rema.loc[X_test_rema.index, 'cluster_id'].values,
    'observed_par_lai': y_test_rema.values,
    'predicted_par_lai': y_pred_test_rema_knn.round(3),
    'residual': (y_pred_test_rema_knn - y_test_rema.values).round(3)
})
print('=== Per-plot residuals (Rema-Kalenga test set, KNN, RFE features) ===')
print(residuals_df_rema_knn.to_string(index=False))
print()

plt.figure(figsize=(6, 6))
plt.scatter(y_test_rema, y_pred_test_rema_knn, color='crimson', edgecolor='black', s=80, label='Test plots')
lims = [min(y_test_rema.min(), y_pred_test_rema_knn.min()) - 0.3,
        max(y_test_rema.max(), y_pred_test_rema_knn.max()) + 0.3]
plt.plot(lims, lims, 'r--', label='1:1 line')
plt.xlabel('Observed PAR LAI')
plt.ylabel('Predicted PAR LAI')
plt.title(f'Rema-Kalenga (KNN, RFE features): Observed vs Predicted PAR LAI (Test Set)\n'
          f'R² = {metrics_test_rema_knn["R2"]}, RMSE = {metrics_test_rema_knn["RMSE"]}')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# NAIVE SPATIAL TRANSFER: KNP (TRANSFER TARGET)
# ============================================================
knp_boundary = ee.FeatureCollection('projects/osmgee/assets/KNP')
knp_sites = ee.FeatureCollection('projects/osmgee/assets/SitesLocation')

print('KNP boundary features:', knp_boundary.size().getInfo())
print('KNP sites features:', knp_sites.size().getInfo())

def clean_feature_knp(f):
    lon = ee.Number(f.get('XCoord'))
    lat = ee.Number(f.get('YCoord'))
    par_lai = ee.Number(f.get('MEAN_PAR_L'))
    cluster_id = f.get('ClusterID')

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'cluster_id': cluster_id,
            'par_lai': par_lai,
            'longitude': lon,
            'latitude': lat
        }
    )

knp_sites_clean = knp_sites.map(clean_feature_knp)

print('Cleaned feature count:', knp_sites_clean.size().getInfo())
print('Sample feature:', knp_sites_clean.first().getInfo())

knp_sites_squares = knp_sites_clean.map(make_square_plot)
print('Sample square geometry (KNP):', knp_sites_squares.first().geometry().getInfo())

# KNP field campaign window (March 2022)
knp_apply_start = ee.Date('2022-01-01')
knp_apply_end = ee.Date('2022-03-31')

knp_s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(knp_boundary) \
    .filterDate(knp_apply_start, knp_apply_end) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

print('Number of available Sentinel-2 scenes (Jan-Mar 2022, KNP):', knp_s2_collection.size().getInfo())

knp_image_list = knp_s2_collection.toList(knp_s2_collection.size())
n_knp = knp_s2_collection.size().getInfo()
for i in range(n_knp):
    img = ee.Image(knp_image_list.get(i))
    date = img.date().format('YYYY-MM-dd').getInfo()
    cloud = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    print(f'{date}  —  cloud cover: {cloud:.1f}%')

knp_s2_masked = knp_s2_collection.linkCollection(csPlus, [QA_BAND]).map(mask_s2_clouds)
knp_s2_composite = knp_s2_masked.median().clip(knp_boundary)
knp_s2_with_indices = add_indices(knp_s2_composite)

final_image_knp = knp_s2_with_indices.select(raw_bands + index_bands)
print('Bands in KNP composite:', final_image_knp.bandNames().getInfo())

knp_extracted = final_image_knp.reduceRegions(
    collection=knp_sites_squares,
    reducer=ee.Reducer.mean(),
    scale=10
)

knp_extracted_info = knp_extracted.getInfo()
knp_rows = [f['properties'] for f in knp_extracted_info['features']]
df_knp = pd.DataFrame(knp_rows)

other_cols = [c for c in df_knp.columns if c not in id_cols]
df_knp = df_knp[id_cols + other_cols]

print(f'Extracted {len(df_knp)} plots')
df_knp


# ============================================================
# APPLY FROZEN KNN MODEL TO KNP (NO RETRAINING) — RFE-SELECTED FEATURES ONLY
# IMPORTANT: use the SAME scaler fit on Rema-Kalenga training data (RFE
# features) — never refit the scaler on KNP data, or this stops being a true
# naive transfer test.
# ============================================================
X_knp = df_knp[rfe_selected_features_knn]
X_knp_scaled = pd.DataFrame(
    scaler.transform(X_knp), columns=rfe_selected_features_knn, index=X_knp.index
)
y_knp_true = df_knp['par_lai']

y_knp_pred_knn = final_knn_rema.predict(X_knp_scaled)

metrics_transfer_knp_knn = compute_metrics(
    y_knp_true, y_knp_pred_knn, 'Spatial Transfer: Rema-Kalenga KNN model (RFE features) -> KNP (40 plots)'
)
transfer_metrics_df_knp_knn = pd.DataFrame([metrics_transfer_knp_knn])

print('=== Spatial Transfer Performance (Rema-Kalenga KNN -> KNP, RFE features) ===')
print(transfer_metrics_df_knp_knn.to_string(index=False))
print()

transfer_residuals_df_knp_knn = pd.DataFrame({
    'cluster_id': df_knp['cluster_id'],
    'observed_par_lai': y_knp_true.round(3),
    'predicted_par_lai': y_knp_pred_knn.round(3),
    'residual': (y_knp_pred_knn - y_knp_true).round(3)
})
print('=== Per-plot residuals (KNP, spatial transfer, KNN, RFE features) ===')
print(transfer_residuals_df_knp_knn.to_string(index=False))
print()

plt.figure(figsize=(6, 6))
plt.scatter(y_knp_true, y_knp_pred_knn, color='darkorange', edgecolor='black', s=80, label='KNP plots')
lims = [min(y_knp_true.min(), y_knp_pred_knn.min()) - 0.3,
        max(y_knp_true.max(), y_knp_pred_knn.max()) + 0.3]
plt.plot(lims, lims, 'r--', label='1:1 line')
plt.xlabel('Observed PAR LAI (KNP)')
plt.ylabel('Predicted PAR LAI (Rema-Kalenga KNN model, RFE features)')
plt.title(f'Spatial Transfer (KNN, RFE features): Rema-Kalenga → KNP\n'
          f'R² = {metrics_transfer_knp_knn["R2"]}, RMSE = {metrics_transfer_knp_knn["RMSE"]}')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# EXPORT COMPOSITES TO GOOGLE DRIVE (for later full-map prediction)
# ============================================================
export_task_knp_knn = ee.batch.Export.image.toDrive(
    image=final_image_knp,
    description='KNP_LAI_features_export_knn',
    folder='GEE_exports',
    fileNamePrefix='KNP_features_2022_JanMar',
    region=knp_boundary.geometry(),
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e13
)
export_task_knp_knn.start()
print('KNP export task started. Task ID:', export_task_knp_knn.id)

export_task_rema_knn = ee.batch.Export.image.toDrive(
    image=final_image_rema,
    description='RemaKalenga_LAI_features_export_knn',
    folder='GEE_exports',
    fileNamePrefix='RemaKalenga_features',
    region=rema_boundary.geometry(),
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e13
)
export_task_rema_knn.start()
print('Rema-Kalenga export task started. Task ID:', export_task_rema_knn.id)

print('KNP task status:', export_task_knp_knn.status()['state'])
print('Rema-Kalenga task status:', export_task_rema_knn.status()['state'])

Rema-Kalenga boundary features: 1
Rema-Kalenga sites features: 59
Available fields: ['XCoord', 'ClusterID', 'OBJECTID_1', 'FREQUENCY', 'MEAN_PAR_L', 'ClusterID_', 'MEAN_GAP_F', 'system:index', 'YCoord']
{'ClusterID': 5, 'MEAN_PAR_L': 0.33117018625, 'XCoord': 91.62458, 'YCoord': 24.11235}
{'ClusterID': 27, 'MEAN_PAR_L': 3.94414734825, 'XCoord': 91.6306, 'YCoord': 24.165805}
{'ClusterID': 35, 'MEAN_PAR_L': 2.494383216, 'XCoord': 91.6436025, 'YCoord': 24.1705425}
{'ClusterID': 51, 'MEAN_PAR_L': 3.0017175675, 'XCoord': 91.6253075, 'YCoord': 24.2033275}
{'ClusterID': 3, 'MEAN_PAR_L': 2.8599865912, 'XCoord': 91.632832, 'YCoord': 24.109208}
{'ClusterID': 4, 'MEAN_PAR_L': 2.0680925488, 'XCoord': 91.626706, 'YCoord': 24.110476}
{'ClusterID': 7, 'MEAN_PAR_L': 3.1344433786, 'XCoord': 91.629136, 'YCoord': 24.113164}
{'ClusterID': 8, 'MEAN_PAR_L': 2.1191831826, 'XCoord': 91.630112, 'YCoord': 24.114924}
{'ClusterID': 9, 'MEAN_PAR_L': 3.2432822702, 'XCoord': 91.631906, 'YCoord': 24.116332}
{'ClusterI